# Chapter 8. RAG Application Evaluation

![Evaluation](./imgs/evaluate-end2end-solution.png)

The figure above illustrates the evaluation process for a RAG application.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datetime import datetime
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness, context_recall, faithfulness
from tqdm import tqdm

from utils.utils import neo4j_driver
from ch08_tools import get_answer

## Designing the Benchmark Dataset

The dataset should include diverse questions that challenge different components:
- *Tool selection evaluation*
- *Entity and value mapping*
- *Multistep retrieval scenarios*
- *Edge cases and functional coverage*
- *Conversational usability*

### Coming up with Test Examples

Each example consists of a question and its corresponding ground truth response, ensuring that the system's output can be reliably assessed.

Instead of providing a static string as the expected answer, we can use Cypher queries to define the ground truth dynamically. This approach allows us to retrieve the correct answer directly from the Neo4j graph database, ensuring that our evaluation is based on the most current and accurate data.

## Evaluation

RAGAS is a framework deisgned for evaluating RAG systems.

### Context Recall

**Context recall** measures how many relevant pieces of information were successfully retrieved using the prompt in "Context recall evaluation".

> **Context recall evaluation** \
Goal: Given a context and an answer, analyze each sentence in the answer and classify whether the sentence can be attributed to the given context or not. Use only 'Yes' (1) or 'No' (0) as a binary classification. Output JSON with reasoning.

### Failthfulness

**Faithfulness** evaluates whether the generated response remains factually consistent with the retrieved context.

Faithfulness is assessed using a two-step process:
1. Decompose the answer into atomic statements using the prompt in "Faithfulness statement breakdown", ensuring that each unit of information is clear and self-contained, making verification easier:
    > **Faithfulness statement breakdown** \
    Goal: Given a question and an answer, analyze the complexity of each sentence in the answer. Break down each sentence into one or more fully understandable statements. Ensure that no pronouns are used in any statement. Format the outputs in JSON.
2. Evaluate each statement against the retrieved context using the prompt in "Faithfulness evaluation":
    > **Faithfulness evaluation** \
    Goal: Your task is to judge the faithfulness of a series of statements based on a given context. For each statement, return a verdict as 1 if the statement can be directly inferred from the context or 0 if the statement cannot be directly inferred from the context.

Finally we evaluate answer correctness by comparing the generated answer with the ground truth.

### Answer Correctness

Answer correctness assesses how accurately and completely the response addresses the user query. It considers both factual accuracy and relevance to ensure the response aligns with the intent of the question.

It uses the same process as failthfulness to generate statements and then evaluates them using the prompt in "Answer correctness evaluation":
> **Answer correctness evaluation** \
Goal: Given a ground truth and an answer statement, analyze each statement and classify it into one of the following categories: \
TP (true positive): Statements present in the answer that are also directly supported by one or more statements in the ground truth. FP (false positive): Statements present in the answer but not directly supported by any statement in the ground truth. FN (false negative): Statements found in the ground truth but not present in the answer. \
Each statement can only belong to one of these categories. Provide a reason for each classification.

### Loading the Dataset

In [ ]:
test_data = pd.read_csv("./data/benchmark_data.csv", delimiter=";")
test_data

### Running Evaluation

In [ ]:
answers = []
ground_truths = []
latencies = []
contexts = []

for i, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Processing rows"):
    # Execute the Cypher query to get the ground truth answer from the Neo4j database
    ground_truth, _, _ = neo4j_driver.execute_query(row["cypher"])
    ground_truths.append([str(el.data()) for el in ground_truth])
    start = datetime.now()

    try:
        # Execute the agent to generate a response to the question
        answer, context = get_answer(row["question"])
        context = [el['content'] for el in context]
    except Exception:
        answer, context = None, []

    # Calculate the latency for processing the question and generating an answer
    latencies.append((datetime.now() - start).total_seconds())

    answers.append(answer)
    contexts.append(context)

In [ ]:
# Store the generated answer and the context used for generating the answer
test_data['ground_truth'] = [str(el) for el in ground_truths]
test_data['answer'] = answers
test_data['latency'] = latencies
test_data['retrieved_contexts'] = contexts

In [ ]:
dataset = Dataset.from_pandas(test_data.fillna("I don't know"))
result = evaluate(
    dataset,
    metrics=[
        answer_correctness,
        context_recall,
        faithfulness,
    ],
)
print(result)

In [ ]:
for key in ["answer_correctness", "context_recall", "faithfulness"]:
    test_data[key] = [el[key] for el in result.scores]
test_data